# 03 — Multi-Armed Bandit
**Week 2 | Mathematical Foundations for RL**

The multi-armed bandit is the simplest RL problem: no states, just actions and stochastic rewards.
It directly introduces the **exploration vs exploitation** trade-off.

Each arm k has a true mean reward μ_k ~ N(0,1). We don't know these — we must learn them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

## 1. Environment Setup

In [ ]:
class Bandit:
    def __init__(self, n_arms=10):
        self.n_arms = n_arms
        self.true_means = np.random.normal(0, 1, n_arms)  # hidden
        self.optimal_arm = np.argmax(self.true_means)

    def pull(self, arm):
        """Return a noisy reward for the chosen arm."""
        return np.random.normal(self.true_means[arm], 1.0)

bandit = Bandit(n_arms=10)
print("True arm means (hidden from agent):", np.round(bandit.true_means, 2))
print(f"Optimal arm: {bandit.optimal_arm} (mean={bandit.true_means[bandit.optimal_arm]:.2f})")

## 2. Agent Implementations

In [ ]:
def run_agent(bandit, policy, n_steps=1000):
    n_arms = bandit.n_arms
    Q = np.zeros(n_arms)   # estimated values
    N = np.zeros(n_arms)   # pull counts
    rewards = []
    optimal_actions = []

    for t in range(n_steps):
        arm = policy(Q, N, t)
        reward = bandit.pull(arm)
        N[arm] += 1
        Q[arm] += (reward - Q[arm]) / N[arm]  # incremental mean update
        rewards.append(reward)
        optimal_actions.append(arm == bandit.optimal_arm)

    return np.array(rewards), np.array(optimal_actions)

# --- Policies ---
def greedy(Q, N, t):          return np.argmax(Q)
def epsilon_greedy(eps):      return lambda Q, N, t: np.random.randint(len(Q)) if np.random.rand() < eps else np.argmax(Q)
def ucb(c=2):                 return lambda Q, N, t: np.argmax(Q + c * np.sqrt(np.log(t+1) / (N+1e-9)))

## 3. Compare Strategies over 2000 Runs

In [ ]:
N_RUNS = 2000
N_STEPS = 1000

configs = [
    ('Greedy',         greedy,              'tomato'),
    ('ε-greedy ε=0.1', epsilon_greedy(0.1), 'steelblue'),
    ('ε-greedy ε=0.01',epsilon_greedy(0.01),'seagreen'),
    ('UCB c=2',        ucb(2),              'darkorange'),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, policy, color in configs:
    all_rewards   = np.zeros((N_RUNS, N_STEPS))
    all_optimal   = np.zeros((N_RUNS, N_STEPS))
    for run in range(N_RUNS):
        b = Bandit(n_arms=10)
        r, opt = run_agent(b, policy, N_STEPS)
        all_rewards[run] = r
        all_optimal[run] = opt
    axes[0].plot(all_rewards.mean(axis=0), label=label, color=color, linewidth=1.5)
    axes[1].plot(all_optimal.mean(axis=0)*100, label=label, color=color, linewidth=1.5)

axes[0].set_xlabel('Steps'); axes[0].set_ylabel('Average reward')
axes[0].set_title('Average Reward over Time'); axes[0].legend(fontsize=9)
axes[1].set_xlabel('Steps'); axes[1].set_ylabel('% Optimal action')
axes[1].set_title('% Optimal Action over Time'); axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

## 4. Regret Analysis
Regret = total reward you *could* have earned (always playing optimal) minus what you actually earned.

In [ ]:
b = Bandit(n_arms=10)
best_mean = b.true_means.max()

plt.figure(figsize=(9, 3.5))
for label, policy, color in configs:
    b = Bandit(n_arms=10)
    # fix the same bandit for fair comparison
    b.true_means = np.array([0.2, -0.5, 1.2, 0.8, -0.1, 0.5, -0.8, 0.3, 1.5, 0.0])
    b.optimal_arm = np.argmax(b.true_means)
    best_mean = b.true_means.max()
    rewards, _ = run_agent(b, policy, N_STEPS)
    cumulative_regret = np.cumsum(best_mean - rewards)
    plt.plot(cumulative_regret, label=label, color=color, linewidth=1.5)
plt.xlabel('Steps'); plt.ylabel('Cumulative regret')
plt.title('Cumulative Regret by Strategy')
plt.legend(fontsize=9); plt.tight_layout(); plt.show()

## ✅ Exercises
1. Add a **softmax / Boltzmann** policy: pick arm proportionally to exp(Q/τ) where τ is temperature. Does it outperform ε-greedy?
2. What happens to UCB if you change c from 2 to 0.1 or 10? Re-run and plot.
3. **Challenge**: implement a **non-stationary** bandit where true means drift over time (add random noise each step). Which strategy handles this best?

## Q1
When c is decreased to 0.1, UCB under-explores — the uncertainty bonus becomes too small, causing the agent to behave almost like a greedy policy and risk locking onto a suboptimal arm early. When c is increased to 10, UCB over-explores — the bonus dominates the Q estimates, so the agent keeps pulling uncertain arms even when enough information has already been collected. c = 2 strikes the best balance, yielding the highest average reward and % optimal action over time.

In [ ]:
configs_q2 = [
    ('UCB c=0.1', ucb(0.1), 'gold'),
    ('UCB c=2',   ucb(2),   'darkorange'),
    ('UCB c=10',  ucb(10),  'firebrick'),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, policy, color in configs_q2:
    all_rewards = np.zeros((N_RUNS, N_STEPS))
    all_optimal = np.zeros((N_RUNS, N_STEPS))
    for run in range(N_RUNS):
        b = Bandit(n_arms=10)
        r, opt = run_agent(b, policy, N_STEPS)
        all_rewards[run] = r
        all_optimal[run] = opt
    axes[0].plot(all_rewards.mean(axis=0), label=label, color=color, linewidth=1.5)
    axes[1].plot(all_optimal.mean(axis=0)*100, label=label, color=color, linewidth=1.5)

axes[0].set_xlabel('Steps'); axes[0].set_ylabel('Average reward')
axes[0].set_title('UCB — Effect of c'); axes[0].legend(fontsize=9)
axes[1].set_xlabel('Steps'); axes[1].set_ylabel('% Optimal action')
axes[1].set_title('UCB — % Optimal Action'); axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

## Q3  Non-Stationary Bandit


In [ ]:
class NonStationaryBandit(Bandit):
    def __init__(self, n_arms=10, drift_std=0.01):
        super().__init__(n_arms)
        self.drift_std = drift_std

    def pull(self, arm):
        reward = np.random.normal(self.true_means[arm], 1.0)
        self.true_means += np.random.normal(0, self.drift_std, self.n_arms)
        self.optimal_arm = np.argmax(self.true_means)
        return reward

def run_agent_constant_step(bandit, policy, n_steps=1000, alpha=0.1):
    n_arms = bandit.n_arms
    Q = np.zeros(n_arms)
    N = np.zeros(n_arms)
    rewards, optimal_actions = [], []
    for t in range(n_steps):
        arm = policy(Q, N, t)
        reward = bandit.pull(arm)
        N[arm] += 1
        Q[arm] += alpha * (reward - Q[arm])
        rewards.append(reward)
        optimal_actions.append(arm == bandit.optimal_arm)
    return np.array(rewards), np.array(optimal_actions)

N_STEPS_NS = 5000
fig, ax = plt.subplots(figsize=(9, 3.5))

for label, policy, color in [('ε-greedy (sample-average)', epsilon_greedy(0.1), 'steelblue'),
                               ('ε-greedy (constant α=0.1)', epsilon_greedy(0.1), 'tomato')]:
    all_optimal = np.zeros((500, N_STEPS_NS))
    for run in range(500):
        b = NonStationaryBandit(n_arms=10, drift_std=0.01)
        if 'constant' in label:
            _, opt = run_agent_constant_step(b, policy, N_STEPS_NS, alpha=0.1)
        else:
            _, opt = run_agent(b, policy, N_STEPS_NS)
        all_optimal[run] = opt
    ax.plot(all_optimal.mean(axis=0)*100, label=label, color=color, linewidth=1.5)

ax.set_xlabel('Steps'); ax.set_ylabel('% Optimal action')
ax.set_title('Non-Stationary Bandit: Sample-Average vs. Constant Step-Size')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

Which strategy handles it best? Constant step-size (α = 0.1) — sample-average updates slow down as N grows and fail to track drifting means.